In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
import shutil
import zipfile
from tqdm import tqdm
from pathlib import Path
from yolo_tools import get_yolo_label_df
from data_vis.yolo_vis import yolo_mdet_vis
from yolo2xanylabeling import yolo_to_xanylabeling_dir
from tqdm.notebook import tqdm
from yolo_tools import get_stem2img

In [4]:
root_path = r'E:\data\202502_signboard\data_annotation\ps_data\task'
class_file = r'E:\data\202502_signboard\data_annotation\docs\class_c6.txt'
att_file = r'E:\data\202502_signboard\data_annotation\docs\attribute.yaml'
yolo_merge_dir = os.path.join(root_path, 'merge_dir_0912')
yolo_merge_image_dir = os.path.join(yolo_merge_dir, 'images')
yolo_merge_label_dir = os.path.join(yolo_merge_dir, 'labels')
yolo_merge_label_check_dir = os.path.join(yolo_merge_dir, 'labels_check')
defect_list = ['deformation', 'broken', 'abandonment', 'corrosion']

task_list = [
        'task_0731_0731',
        'task_0725_0805',
        'task_0610_0816',
        'task_0811_0816',
        "task_0806_0821",
        "task_0808_0821",
        "task_0812_0821",
        "task_0819_0821",
        "task_0908_0908",
        "task_404_0912",
        "task_405_0912",
    ]

In [ ]:
def merge_labels(root_path, task_list, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    count = 0
    for idx, task_name in enumerate(task_list):
        input_dir = os.path.join(root_path, task_name, 'yolo_data', 'labels')
        label_list = os.listdir(input_dir)
        count += len(label_list)
        for label_name in tqdm(label_list, desc=f'copy {idx}/{len(task_list)}: {task_name}'):
            input_path = os.path.join(input_dir, label_name)
            output_path = os.path.join(output_dir, label_name)
            shutil.copy(input_path, output_path)
    print(f'copy {len(os.listdir(output_dir))}, repeat {count - len(os.listdir(output_dir))}')


In [ ]:
merge_labels(root_path, task_list, yolo_merge_label_dir)

In [11]:
def att_check(input_dir, output_dir, reorder=True, rm_id=True):
    os.makedirs(output_dir, exist_ok=True)
    count = 0
    label_list = os.listdir(input_dir)
    track_list = []
    for label_name in tqdm(label_list):
        input_label_path = os.path.join(input_dir, label_name)
        output_label_path = os.path.join(output_dir, label_name)
        with open(input_label_path, 'r') as f1, open(output_label_path, 'w') as f2:
            lines = f1.readlines()
            new_lines = []
            for idx, line in enumerate(lines):
                parts = line.strip().split(' ')
                risk_list = parts[2:6]
                if reorder:
                    new_risk_list = [risk_list[3], risk_list[1], risk_list[0], risk_list[2]]
                    parts[2:6] = new_risk_list
                if rm_id:
                    if len(parts) % 2 == 1:
                        parts = parts[:-1]
                        track_list.append(label_name)
                new_line = ' '.join(parts) + '\n'
                if line[2] == '0':
                    lines[idx] = lines[idx][0:2] + '4' + lines[idx][3:]
                    count += 1
                new_lines.append(new_line)
            f2.writelines(new_lines)
        with open(input_label_path, 'w') as f:
            f.writelines(lines)
    print(f'change {count} lines')
    track_list = list(set(track_list))
    print(len(track_list), track_list)


In [12]:
yolo_merge_label_dir = r'E:\data\202502_signboard\data_annotation\ps_data\task\check_0914_labels_0914\check_0914_labels_0914'
yolo_merge_label_check_dir = r'E:\data\202502_signboard\data_annotation\ps_data\task\check_0914_labels_0914\check_0914_labels_0914_check'
att_check(yolo_merge_label_dir, yolo_merge_label_check_dir, reorder=False, rm_id=True)


  0%|          | 0/331 [00:00<?, ?it/s]

change 0 lines
299 ['DA5148683_20250711144854100.txt', 'input_1_DA5148683_20250709161532900.txt', 'DA5148683_20250722114921600.txt', 'DA5148683_20250711145351999.txt', 'DA5324655_20250722155317999.txt', 'cam_DA4930148_cam_image_20250722114213600.txt', 'cam_DA5324645_cam_DA5324645_20250610132631916.txt', 'DA5148680_20250812140354599.txt', 'cam_DA5148683_cam_image_20250627144116800.txt', 'cam_DA5148683_cam_image_20250722114922099.txt', 'DA5148680_DA5148680_20241013015617400.txt', 'DA5148683_20250722154446800.txt', 'cam_DA5148680_cam_image_20250702160858900.txt', 'cam_DA5148680_cam_image_20250722151913500.txt', 'cam_DA5148680_cam_image_20250704160131599.txt', 'cam_DA5148680_cam_image_20250702161254999.txt', 'DA5148680_20250716140857700.txt', 'DA5324645_20250812141754900.txt', 'DA5148680_20250722151911099.txt', 'cam_DA4930148_cam_image_20250709164524600.txt', 'DA5148680_20250722114440799.txt', 'DA5324655_DA5324655_20250620072522100.txt', 'cam_DA5148683_cam_image_20250702160851500.txt', 'ca

In [8]:
def att_second_check(input_dir):
    label_list = os.listdir(input_dir)
    track_list = []
    for label_name in tqdm(label_list):
        input_label_path = os.path.join(input_dir, label_name)
        with open(input_label_path, 'r') as f1:
            lines = f1.readlines()
            new_lines = []
            for idx, line in enumerate(lines):
                parts = line.strip().split(' ')
                risk_list = parts[2:6]
                new_risk_list = [risk_list[3], risk_list[1], risk_list[0], risk_list[2]]
                parts[2:6] = new_risk_list
                if len(parts) % 2 == 1:
                    parts = parts[:-1]
                    track_list.append(label_name)
                    break
    track_list = list(set(track_list))
    print(len(track_list), track_list)


In [9]:
att_second_check(yolo_merge_label_check_dir)

  0%|          | 0/331 [00:00<?, ?it/s]

0 []
